# ML Lab Part II – Model Training & Hyperparameter Optimization

## Goal
Train different classification algorithms on the feature dataset from Part I
and optimize their hyperparameters.

## Workflow
```
Load data → Train/Test-Split → Train classifier →
Measure performance → Optimize hyperparameters
```

**Prerequisite:** `data/feature_data.csv` from Part I must exist.

## Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import warnings; warnings.filterwarnings('ignore')

DATA_DIR = Path("data")
ACTIVITIES = {1: "Walking", 2: "Standing", 3: "Squats", 4: "SpinAround"}

---
## Task 1 – Load data

Load the feature dataset from Part I. We start with the one that provides only the 4 activities. 

**Hint:** `pd.read_csv()`, vizualize at the table, then separate features (X) and labels (y).
The `label` column contains the class labels.

In [7]:
# TODO: load feature_data.csv
# X = all columns except 'label'
# y = 'label' column
df = pd.read_csv(DATA_DIR/"feature_data.csv")
df = pd.DataFrame(df)
X  = df.drop(columns=["Label"])
display(X)
y  = df["Label"]

print(f"Dataset: {X.shape[0]} Samples, {X.shape[1]} Features")
print(f"Classes: {np.unique(y)}")

,acc_x_mean,acc_x_std,acc_x_min,acc_x_max,acc_y_mean,acc_y_std,acc_y_min,acc_y_max,acc_z_mean,acc_z_std,...,gyro_x_min,gyro_x_max,gyro_y_mean,gyro_y_std,gyro_y_min,gyro_y_max,gyro_z_mean,gyro_z_std,gyro_z_min,gyro_z_max
0,1.847395,5.095993,-6.9059,9.4052,-2.030817,3.630154,-8.6967,3.7232,0.681831,1.933812,...,-0.9525,1.6704,-0.467312,1.697025,-5.8773,1.7831,0.112597,2.411614,-3.9315,4.0090
1,1.798158,4.557352,-6.9059,8.1160,-1.679730,3.893862,-8.6967,3.6836,0.191241,1.024300,...,-0.9525,1.4390,0.105650,1.160391,-2.7875,1.6881,-0.041758,2.317314,-3.9315,3.3988
2,1.433537,4.934589,-7.0327,8.1160,-2.687134,4.721251,-11.1530,3.6836,0.517114,1.112028,...,-1.8564,1.7900,-0.006725,1.372668,-3.8767,1.6490,-0.206572,2.392862,-3.6437,5.7053
3,1.675386,5.193139,-7.0327,9.2318,-2.821614,4.666588,-11.1530,4.0556,0.879195,1.144199,...,-1.8564,1.7900,0.054781,1.489084,-3.8767,2.8173,0.272145,2.495037,-3.5247,5.7053
4,2.499689,4.784814,-6.6113,9.2318,-2.102866,4.649352,-11.5201,4.0556,1.250675,0.895729,...,-1.5620,1.6248,0.039753,1.271210,-2.0823,2.8173,-0.433359,2.265394,-3.2912,5.3870
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
556,0.158261,2.400805,-6.0255,6.4617,-0.118253,1.706517,-9.9599,4.5861,1.126252,2.277579,...,-1.4917,2.2664,2.612903,1.051718,-0.5211,4.1637,-0.063252,0.567983,-1.3382,1.1595
557,0.175042,2.435855,-6.0255,6.4617,-0.077220,1.787776,-9.9599,4.5861,0.767881,1.974443,...,-1.6266,1.8665,2.644397,0.689493,0.9414,4.1637,-0.135692,0.528006,-1.3382,1.1595
558,0.239236,2.083462,-4.6657,3.6195,-0.019192,1.191194,-4.6306,2.4921,0.875166,1.746633,...,-1.6266,1.8262,2.627823,0.738691,0.9414,4.5031,-0.036097,0.477579,-1.1733,1.0352
559,0.232800,2.240956,-6.2748,3.4289,-0.015367,1.307462,-4.9004,2.1604,1.229552,1.631468,...,-1.6793,1.9730,2.878259,0.647271,1.1776,4.5031,-0.066144,0.609325,-1.3605,1.0405


Dataset: 561 Samples, 24 Features
Classes: [1. 2. 3. 4.]


---
## Task 2 – Train/Test-Split

Split the dataset into **85% training** and **15% test**.
The test split is not used for training. In this lab, we will also inspect it once in Task 3 to compare it with cross-validation, but do not use it to choose or tune models.

**Hints:**
- use `stratify=y` to ensure that all classes are distributed evenly

In [11]:
# TODO: split the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.15, random_state=3,stratify=y)
print(f"Training:  {X_train.shape[0]} Samples")
print(f"Test:      {X_test.shape[0]} Samples")

Training:  476 Samples
Test:      85 Samples


---
## Task 3 – Train and compare classifiers

Train at least the following three classifiers on the training data:

| Class | sklearn |
|---|---|
| Decision Tree | `DecisionTreeClassifier()` |
| k-Nearest Neighbors | `KNeighborsClassifier()` |
| Random Forest | `RandomForestClassifier()` |

For each classifier:
- **a)** Training on `X_train` / `y_train`
- **b)** Training-Accuracy (Resubstitution) with `model.score(X_train, y_train)`
- **c)** Cross-Validation-Accuracy (5-fold) with `cross_val_score(..., cv=5)`
- **d)** Test-Accuracy with `model.score(X_test, y_test)`

**Own research:** We did not cover `cross_val_score` in detail in the lecture. Familiarize yourself with the goal of **k-fold cross validation**: Why is the training data split into several folds, why is a fresh model trained for each fold, and why can this give a better estimate than only checking the training accuracy?

**Hint:** `cross_val_score` returns an array; use the mean with `.mean()`.
Compare the cross-validation accuracy with the test accuracy. This test-score comparison is included here for learning purposes only. In a clean ML workflow, the test set should not be used to choose between models or tune hyperparameters.

Is there a difference between cross-validation accuracy and test accuracy in your case? If yes, what could explain it?

<div style="border: 1px solid #999; padding: 10px; min-height: 120px; background: #fafafa;">
<strong>Your explanation:</strong><br><br>
Write your explanation here.
</div>

In [ ]:
models = {
    "Decision Tree":  DecisionTreeClassifier(random_state=42),
    "k-NN":           KNeighborsClassifier(),
    "Random Forest":  RandomForestClassifier(random_state=42),
}

for i in models.values():
    current_algorithm = i
    current_algorithm.fit(X_train,y_train)
    current_algorithm.score(X_train,y_train)


    #she asked do you have vs .. she replied no


#Train and evaluate all models An Output similar to this (values might differ)
#Decision Tree         Resub: 1.000  CV: 0.982 Test: 0.960
#k-NN                  Resub: 0.996  CV: 0.993 Test: 1.000
#Random Forest         Resub: 1.000  CV: 0.993 Test: 1.000

---
## Task 4 – Compare with another group's data

Ask another group for their feature dataset from Part I and use it as an additional check: Are the cross-validation results from your own training data comparable to the accuracy on data recorded by another group?

Use this exact filename so everyone follows the same convention:

```text
data/feature_data_external.csv
```

The file must have the same structure as your own `feature_data.csv`: the same feature columns and a `label` column with the same activity labels.

Evaluate the models from Task 3 on this external dataset and compare the result with the CV accuracy from Task 3.

If the external accuracy is much lower than the CV accuracy, investigate the reason: Join your dataset and the external dataset, train new classifiers on the joined data, and compare whether the results improve. This can show whether the original model learned patterns that were too specific to your own group's recordings.

<div style="border: 1px solid #999; padding: 10px; min-height: 120px; background: #fafafa;">
<strong>Your explanation:</strong><br><br>
Are the external-data accuracies comparable to the cross-validation accuracies? Does training on the joined dataset improve the result? What could explain the difference?
</div>

In [ ]:
external_file = DATA_DIR / "feature_data_external.csv"

# TODO: load the external feature dataset
# TODO: create X_external and y_external
# TODO: evaluate each model from Task 3 on the external dataset
# TODO: compare external accuracy with the CV accuracy from Task 3
# TODO: join your dataset and the external dataset
# TODO: train new classifiers on the joined dataset
# TODO: compare the new CV accuracy with the previous external accuracy

---
## Task 5 – Train classifiers for all 8 activities

In Part I, you created a larger feature dataset with all 8 activity classes. Now repeat the training and comparison from Task 3 with this larger dataset.

Use this exact filename:

```text
data/large_feature_data.csv
```

The expected labels are:

| Label | Activity |
|---|---|
| 1 | Walking |
| 2 | Standing |
| 3 | Squats |
| 4 | SpinAround |
| 5 | Boxing |
| 6 | Jumping |
| 7 | KneeRaise |
| 8 | WalkingBackward |

Train the same classifiers as before and compare training accuracy, 5-fold cross-validation accuracy, and test accuracy. Does the classification problem become harder when all 8 classes are included?

<div style="border: 1px solid #999; padding: 10px; min-height: 120px; background: #fafafa;">
<strong>Your explanation:</strong><br><br>
Compare the 8-class results with the 4-class results. Which classes or models seem more difficult, and why?
</div>

In [ ]:
large_file = DATA_DIR / "large_feature_data.csv"

# TODO: load large_feature_data.csv
# TODO: create X_large and y_large
# TODO: split into training and test data
# TODO: train the same classifiers as in Task 3
# TODO: compare resubstitution, CV, and test accuracy

---
## Task 6 – Plot a prediction crosstab

Accuracy gives only one number. To understand which activities are confused with each other, create a crosstab of true labels vs. predicted labels for the 8-class test set from Task 5.

Use one trained model from Task 5, for example the Random Forest model. Then:

- predict the labels for `X_test_large`
- create a crosstab with `pd.crosstab(...)`
- display the crosstab as a table
- plot the same values as a heatmap

<div style="border: 1px solid #999; padding: 10px; min-height: 120px; background: #fafafa;">
<strong>Your explanation:</strong><br><br>
Which activities are confused most often? Does this match your expectation from the movements?
</div>

In [ ]:
# TODO: select one model from Task 5, for example Random Forest
# TODO: predict labels for X_test_large
# TODO: create a crosstab with pd.crosstab(...)
# TODO: display the crosstab
# TODO: plot the crosstab as a heatmap

---
## Task 7 – Automatic hyperparameter optimization

So far, you trained classifiers mostly with their default settings. Many machine-learning algorithms have **hyperparameters**: settings that are chosen before training and can strongly influence the result. Examples are the maximum depth of a decision tree, the number of neighbors in k-NN, or the number and depth of trees in a random forest.

Trying these settings manually is slow and can easily become inconsistent. In this task, you let scikit-learn test several hyperparameter combinations automatically. The important point is that the search uses **cross-validation on the training data**. The test set should not be used to decide which hyperparameters are best.

You will optimize three algorithms and compare two common search strategies:

| Algorithm | Search strategy | Example hyperparameters |
|---|---|---|
| Decision Tree | `GridSearchCV` | `max_depth`, `min_samples_split`, `criterion` |
| k-NN | `RandomizedSearchCV` | `n_neighbors`, `weights`, `metric` |
| Random Forest | `RandomizedSearchCV` | `n_estimators`, `max_depth`, `min_samples_split` |

After the searches, compare the best cross-validation score of each optimized model with the default models from Task 3.

You will use these two search strategies:

### a) Grid Search
Searches **all** combinations in the defined grid.
```python
GridSearchCV(estimator, param_grid, cv=5, scoring='accuracy')
```

### b) Random Search
Samples **random** combinations from the search space (faster for large spaces).
```python
RandomizedSearchCV(estimator, param_distributions, n_iter=20, cv=5)
```

**Hints:**
- `.best_params_` shows the best parameters found
- `.best_score_` shows the best CV accuracy
- `.fit(X_train, y_train)` starts the search

In [ ]:
# TODO: Grid Search for Decision Tree
param_grid_dt = {
    "max_depth": [3, 5, 10, None],
    "min_samples_split": [2, 5, 10],
    "criterion": ["gini", "entropy"],
}
# grid_dt = GridSearchCV(...)
# grid_dt.fit(...)

# TODO: Random Search for k-NN
param_dist_knn = {
    "n_neighbors": range(1, 30),
    "weights": ["uniform", "distance"],
    "metric": ["euclidean", "manhattan"],
}
# rand_knn = RandomizedSearchCV(...)
# rand_knn.fit(...)

# TODO: Random Search for Random Forest
param_dist_rf = {
    "n_estimators": [50, 100, 200],
    "max_depth": [5, 10, None],
    "min_samples_split": [2, 5],
}
# rand_rf = RandomizedSearchCV(...)
# rand_rf.fit(...)


# TODO: print results and compare them with default models

---
## Task 8 – Evaluate the best model on the test set

Choose the best model from Task 7 and evaluate it on the test set.
In a clean ML workflow, the test set should be used only **once**, at the end. In this lab, you already inspected it in Task 3 for comparison; do not use the Task 3 test scores for hyperparameter tuning.

**Hint:** `best_estimator_` contains the already trained best model.

In [ ]:
# TODO: select the best model and evaluate it on X_test
# best_model = grid_dt.best_estimator_  (or rand_knn / rand_rf)
# test_acc = accuracy_score(y_test, best_model.predict(X_test))
# print(f"Test-Accuracy: {test_acc:.3f}")